In [0]:
# Import libraries
from pyspark.sql.functions import col, when, sum as spark_sum
from delta.tables import DeltaTable

# Read master CSV
master_path = "file:/Workspace/week 7/customer_master.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(master_path)
)

# Rename columns
df = df.toDF(
    "row_id",
    "order_id",
    "order_date",
    "ship_date",
    "ship_mode",
    "customer_id",
    "customer_name",
    "segment",
    "country",
    "city",
    "state",
    "postal_code",
    "region",
    "product_id",
    "category",
    "sub_category",
    "product_name",
    "sales",
    "quantity",
    "discount",
    "profit",
    "total_amount"
)

print(f"Rows : {df.count()}")
print(f"Columns : {len(df.columns)}")

df.printSchema()
df.show(5, truncate=False)

Rows : 9994
Columns : 22
root
 |-- row_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- profit: double (nullable = true)
 |-- total_amount: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+--------

In [0]:
# Create Delta table
table_name = "workspace.default.superstore_master"

spark.sql(f"DROP TABLE IF EXISTS {table_name}")

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)

print("Delta table created.")
print("Rows :", spark.table(table_name).count())

Delta table created.
Rows : 9994


In [0]:
# Count null values
null_df = df.select([
    spark_sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

null_df.show(truncate=False)

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+------------+
|row_id|order_id|order_date|ship_date|ship_mode|customer_id|customer_name|segment|country|city|state|postal_code|region|product_id|category|sub_category|product_name|sales|quantity|discount|profit|total_amount|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+------------+
|0     |0       |0         |0        |0        |0          |0            |0      |0      |0   |0    |0          |0     |0         |0       |0           |0           |0    |0       |0       |0     |0           |
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+---

In [0]:
# Replace null postal codes
spark.sql("""
UPDATE workspace.default.superstore_master
SET postal_code = 0
WHERE postal_code IS NULL
""")

print("Postal codes updated.")

Postal codes updated.


In [0]:
# Check duplicate row_ids
dup_count = (
    spark.table("workspace.default.superstore_master")
    .groupBy("row_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate Rows :", dup_count)

if dup_count > 0:

    df = (
        spark.table("workspace.default.superstore_master")
        .dropDuplicates(["row_id"])
    )

    spark.sql("DROP TABLE workspace.default.superstore_master")

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable("workspace.default.superstore_master")

    print("Duplicates removed.")

else:

    print("No duplicates found.")

Duplicate Rows : 0
No duplicates found.


In [0]:
# Display cleaned table
df = spark.table("workspace.default.superstore_master")

print("Rows :", df.count())

df.show(5, truncate=False)

Rows : 9994
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+--------+--------+--------+--------+------------+
|row_id|order_id      |order_date|ship_date |ship_mode     |customer_id|customer_name  |segment  |country      |city           |state     |postal_code|region|product_id     |category       |sub_category|product_name                                               |sales   |quantity|discount|profit  |total_amount|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+--------+--------+--------+--------+------------+
|1     |CA-2016-152156|2016-11-08|2016-11-11|Seco

In [0]:
# Read incremental CSV
inc_path = "file:/Workspace/week 7/customer_incremental.csv"

inc_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(inc_path)
)

# Rename columns
inc_df = inc_df.toDF(
    "row_id",
    "order_id",
    "order_date",
    "ship_date",
    "ship_mode",
    "customer_id",
    "customer_name",
    "segment",
    "country",
    "city",
    "state",
    "postal_code",
    "region",
    "product_id",
    "category",
    "sub_category",
    "product_name",
    "sales",
    "quantity",
    "discount",
    "profit",
    "total_amount"
)

print("Incremental Rows :", inc_df.count())

inc_df.show(5, truncate=False)

Incremental Rows : 250
+------+--------------+----------+----------+--------------+-----------+----------------+---------+-------------+-------+--------+-----------+-------+---------------+---------------+------------+--------------------------------------------------+------+--------+--------+--------+------------+
|row_id|order_id      |order_date|ship_date |ship_mode     |customer_id|customer_name   |segment  |country      |city   |state   |postal_code|region |product_id     |category       |sub_category|product_name                                      |sales |quantity|discount|profit  |total_amount|
+------+--------------+----------+----------+--------------+-----------+----------------+---------+-------------+-------+--------+-----------+-------+---------------+---------------+------------+--------------------------------------------------+------+--------+--------+--------+------------+
|9795  |CA-2014-127166|2014-05-21|2014-05-23|Second Class  |KH-16360   |Katherine Hughes|Consum

In [0]:
# Count updates and inserts
update_count = (
    inc_df.join(
        spark.table("workspace.default.superstore_master"),
        on="row_id",
        how="inner"
    )
    .count()
)

insert_count = (
    inc_df.join(
        spark.table("workspace.default.superstore_master"),
        on="row_id",
        how="left_anti"
    )
    .count()
)

print("Rows to Update :", update_count)
print("Rows to Insert :", insert_count)

Rows to Update : 200
Rows to Insert : 50


In [0]:
# Store current row count
before_count = spark.table("workspace.default.superstore_master").count()

print("Before Merge :", before_count)

Before Merge : 9994


In [0]:
# Merge incremental data
delta_df = DeltaTable.forName(
    spark,
    "workspace.default.superstore_master"
)

(
    delta_df.alias("target")
    .merge(
        inc_df.alias("source"),
        "target.row_id = source.row_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

after_count = spark.table("workspace.default.superstore_master").count()

print("After Merge :", after_count)
print("Inserted :", after_count - before_count)

After Merge : 10044
Inserted : 50


In [0]:
# Validate row count
print("Before :", before_count)
print("After  :", after_count)

assert after_count - before_count == 50

print("Row count validation passed.")

Before : 9994
After  : 10044
Row count validation passed.


In [0]:
# Verify duplicates
dup_count = (
    spark.table("workspace.default.superstore_master")
    .groupBy("row_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate Rows :", dup_count)

assert dup_count == 0

print("Duplicate validation passed.")

Duplicate Rows : 0
Duplicate validation passed.


In [0]:
# Display updated rows
sample_ids = [
    row.row_id
    for row in inc_df.select("row_id").limit(5).collect()
]

(
    spark.table("workspace.default.superstore_master")
    .filter(col("row_id").isin(sample_ids))
    .select(
        "row_id",
        "sales",
        "quantity",
        "total_amount"
    )
    .show(truncate=False)
)

+------+------+--------+------------+
|row_id|sales |quantity|total_amount|
+------+------+--------+------------+
|9795  |20.06 |3.0     |60.18       |
|9796  |4.18  |3.0     |12.54       |
|9797  |11.4  |2.0     |22.8        |
|9798  |258.71|2.0     |517.42      |
|9799  |29.01 |4.0     |116.04      |
+------+------+--------+------------+



In [0]:
# Check new records
new_rows = (
    spark.table("workspace.default.superstore_master")
    .filter(col("order_id").startswith("INCR-2026"))
    .count()
)

print("New Rows :", new_rows)

assert new_rows == 50

print("Insert validation passed.")

New Rows : 50
Insert validation passed.


In [0]:
# Show Delta history
delta_df = DeltaTable.forName(
    spark,
    "workspace.default.superstore_master"
)

delta_df.history().show(truncate=False)

+-------+-------------------+--------------+--------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
# Display final table
df = spark.table("workspace.default.superstore_master")

print("Final Rows :", df.count())
print("Columns :", len(df.columns))

df.printSchema()

df.show(10, truncate=False)

Final Rows : 10044
Columns : 22
root
 |-- row_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- profit: double (nullable = true)
 |-- total_amount: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+-

In [0]:
# Show summary statistics
df.describe(
    "sales",
    "quantity",
    "discount",
    "profit",
    "total_amount"
).show()

+-------+------------------+------------------+-------------------+------------------+------------------+
|summary|             sales|          quantity|           discount|            profit|      total_amount|
+-------+------------------+------------------+-------------------+------------------+------------------+
|  count|             10044|             10044|              10044|             10044|             10044|
|   mean|234.58167220568598| 5.857271149241816|0.31536768663411374| 28.61097667264055|1138.6402154520108|
| stddev|  630.516983064894|25.638476406216036| 3.3066838312407514|233.86482292997545|3891.7668242251207|
|    min|          10/Pack"|      1040 sheets"|            30/Box"|         -6599.978|         -292.9872|
|    max|            999.98|            98.352|             98.352|          8399.976|         135830.88|
+-------+------------------+------------------+-------------------+------------------+------------------+



In [0]:
# Count records by category
df.groupBy("category") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

+---------------+-----+
|       category|count|
+---------------+-----+
|Office Supplies| 6062|
|      Furniture| 2126|
|     Technology| 1856|
+---------------+-----+



In [0]:
# Display merged records
df.select(
    "row_id",
    "customer_name",
    "category",
    "sales",
    "profit"
).show(10, truncate=False)

+------+---------------+---------------+--------+--------+
|row_id|customer_name  |category       |sales   |profit  |
+------+---------------+---------------+--------+--------+
|1     |Claire Gute    |Furniture      |261.96  |41.9136 |
|2     |Claire Gute    |Furniture      |731.94  |219.582 |
|3     |Darrin Van Huff|Office Supplies|14.62   |6.8714  |
|4     |Sean O'Donnell |Furniture      |957.5775|-383.031|
|5     |Sean O'Donnell |Office Supplies|22.368  |2.5164  |
|6     |Brosina Hoffman|Furniture      |48.86   |14.1694 |
|7     |Brosina Hoffman|Office Supplies|7.28    |1.9656  |
|8     |Brosina Hoffman|Technology     |907.152 |90.7152 |
|9     |Brosina Hoffman|Office Supplies|18.504  |5.7825  |
|10    |Brosina Hoffman|Office Supplies|114.9   |34.47   |
+------+---------------+---------------+--------+--------+
only showing top 10 rows


In [0]:
# Summary of all the operations
print("=" * 50)
print("Delta Lake MERGE Assignment Summary")
print("=" * 50)
print(f"Initial Rows      : {before_count}")
print(f"Final Rows        : {after_count}")
print(f"Updated Records   : {update_count}")
print(f"Inserted Records  : {insert_count}")
print(f"Duplicate Records : {dup_count}")
print("=" * 50)
print("Assignment Completed Successfully")

Delta Lake MERGE Assignment Summary
Initial Rows      : 9994
Final Rows        : 10044
Updated Records   : 200
Inserted Records  : 50
Duplicate Records : 0
Assignment Completed Successfully
